# 00 – Setup check
Goal of day 1: confirm that Python talks to Google Earth Engine and draw the first map of the study area.

If every cell runs without errors, the setup is done.

In [1]:
import sys
sys.path.append("..")  # lets the notebook import config.py and src/

import ee
import geemap
from config import GEE_PROJECT, AOI_BBOX
from src.s2_utils import s2_clean, summer_composite

C:\Users\ivana\miniforge3\envs\atlas\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 1. Connect to Earth Engine
The first time, `ee.Authenticate()` opens a browser window to log in with your Google account. After that you can comment it out.

In [2]:
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print("Earth Engine OK")

Earth Engine OK


## 2. Study area

In [3]:
aoi = ee.Geometry.Rectangle(AOI_BBOX)
area_km2 = aoi.area(maxError=1).divide(1e6).getInfo()
print(f"AOI area: {area_km2:.0f} km²")

AOI area: 1458 km²


## 3. How many clear Sentinel-2 images do we have?
Summer 2025, cloud-masked with Cloud Score+.

In [4]:
col = s2_clean(aoi, "2025-06-01", "2025-08-31")
print("Sentinel-2 images in summer 2025:", col.size().getInfo())

Sentinel-2 images in summer 2025: 110


## 4. First map: true color and NDVI, summer 2025

In [5]:
img = summer_composite(aoi, 2025)

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(img, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Sentinel-2 true color 2025")
m.addLayer(img.select("NDVI"), {"min": 0, "max": 0.9, "palette": ["#a0522d", "#f5f5dc", "#2e8b57", "#004d1a"]}, "NDVI 2025")
m.addLayer(ee.FeatureCollection([ee.Feature(aoi)]).style(color="red", fillColor="00000000"), {}, "AOI")
m

Map(center=[49.49514050255065, 11.125000000000297], controls=(WidgetControl(options=['position', 'transparent_…

## 5. Quick check: mean NDVI per year (Reichswald preview)
A first look at the question of case study 1. Values drop in dry years (2018, 2019, 2022)?

In [6]:
for year in [2018, 2020, 2022, 2025]:
    mean_ndvi = (summer_composite(aoi, year).select("NDVI")
                 .reduceRegion(ee.Reducer.mean(), aoi, scale=100, maxPixels=1e9)
                 .get("NDVI").getInfo())
    print(year, round(mean_ndvi, 3))

2018 0.579
2020 0.604
2022 0.589
2025 0.642


## Done?
- [ ] Earth Engine connected
- [ ] Map shows the three cities and the Reichswald
- [ ] NDVI values printed

Next (day 2): define the Reichswald forest mask and build the 2017–2025 NDVI/NDMI time series.